# Field validation — `surface_wind` (SURF pipeline)

End-to-end validation of every calculated field in the **surface_wind** surface subset: the notebook RUNs the pipeline, LOADs its own output, and validates each field with dependency-chain maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `surface_wind` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Wind-stress diagnostics.  τ comes from the LLC_SURF S3 forcing stores at this date (OSN llc_wind kerchunk ends 2012-07-15); the 20121109T12 LLC_SURF raw data is transferred and available.

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

Note: at 2012-11-09 the wind variables come from the LLC_SURF S3 stores (outside the OSN llc_wind window).  The 20121109T12 LLC_SURF raw data has been TRANSFERRED (plan §9 item resolved 2026-07-31), so no OSN wind data is needed.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "surface_wind"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
# get_subset_definition already folds per-pipeline extras (e.g.
# oceQnet for SURF) into model_data_feature_channels.
CHANNELS = (list(defn["model_data_feature_channels"])
            + list(defn["compute_features_channels"]))

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `surface_wind`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`):

raw (SURF only) — `oceQnet`; computed — `oceTAUX`, `oceTAUY`, `wind_stress_curl`, `ekman_pumping`, `u_ekman`, `v_ekman`

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert set(reader.channel_names) == set(CHANNELS), (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel set matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| oceTAUX | N m⁻² | staggered τ → interp + CS/SN rotation → eastward | raw oceTAUX, oceTAUY; CS, SN | `calculate_fields.geographic_wind_stress` |
| oceTAUY | N m⁻² | same → northward | raw oceTAUX, oceTAUY; CS, SN | `calculate_fields.geographic_wind_stress` |
| wind_stress_curl | N m⁻³ | ∇×τ (native gradients of rotated τ) | oceTAUX, oceTAUY; grid metrics | `calculate_fields.wind_stress_curl` |
| ekman_pumping | m s⁻¹ | w_E = curl(τ)/(ρ₀ f); NaN at equator | wind_stress_curl, f | `calculate_fields.ekman_pumping` |
| u_ekman | m² s⁻¹ | Ekman transport (τ/ρ₀f, 90° right of τ in NH) | oceTAUX, oceTAUY, f | `calculate_fields.ekman_transport` |
| v_ekman | m² s⁻¹ | same, meridional component | oceTAUX, oceTAUY, f | `calculate_fields.ekman_transport` |
| oceQnet | W m⁻² | model output (SURF pipeline only) | — | LLC_SURF store |

Intermediate plotted in dependency columns: `coriolis_f`
(f = 2Ω·sin lat), recomputed in-notebook on the rect grid — an
independent check on the pipeline's
`calculate_fields.coriolis_parameter`.

Processing operations: land masking; staggered→tracer interpolation +
CS/SN rotation (τ); native-grid differentiation (curl; halo rim);
f-division (ekman fields — equatorial NaNs expected); face→lat-lon
stitching; global downsampling.

### In-notebook extras: coriolis_f + surface currents

`coriolis_f` is recomputed on the rect grid (same equation as `calculate_fields.coriolis_parameter`; independent cross-check), masked to ocean pixels.  Surface currents `U`, `V` are loaded from the **native_fields store of the same run/date** as COMPANION columns for the rotated wind stress (not dependencies): expect broad stress/current alignment under the trades and westerlies with Ekman deflection, and WBC jets that are NOT wind-aligned.  Run native_fields.ipynb Section 1 first if that store is missing.

In [ ]:
# coriolis_f on the rect grid (same equation as
# calculate_fields.coriolis_parameter), ocean-masked via oceTAUX.
from dbof.plotting import regions
from dbof.preprocessing.physical_constants import OMEGA_EARTH

f_rect = (2.0 * OMEGA_EARTH
          * np.sin(np.radians(YC))).astype("float32")
_tau = reader.get_channel_snapshot("oceTAUX")
f_rect = np.where(np.isfinite(_tau), f_rect, np.nan)
del _tau

region_arrays = {}
region_arrays["coriolis_f"] = regions.select_all_regions(
    f_rect, XC, YC)
del f_rect
print("coriolis_f ready (rect-grid cross-check)")

# Surface currents U, V from the native_fields store of the SAME
# run/date — companion columns for the rotated wind stress (NOT
# dependencies; cross-subset physical comparison).  Requires the
# native_fields store for this run_id/date (generated by
# native_fields.ipynb).
defn_nf = get_subset_definition(PIPELINE, "native_fields")
reader_nf = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn_nf["dataset_name"], date_prefix=DATE_PREFIX,
    fs=fs,
)
for _ch in ["U", "V"]:
    _arr = reader_nf.get_channel_snapshot(_ch)
    region_arrays[_ch] = regions.select_all_regions(_arr, XC, YC)
    del _arr
print("U, V loaded from native_fields store (same run/date)")

In [ ]:
# Slice the STORE channels to the validation domains (live fields, if
# any, were sliced in the previous cell).  Full-res arrays released
# immediately after slicing.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps

CMAP_CFG, DIVERGING = load_field_cmaps()

# region_arrays[field][region] = (x, y, arr)
region_arrays = globals().get("region_arrays", {})
SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
for ch in CHANNELS:
    arr = reader.get_channel_snapshot(ch)
    region_arrays[ch] = regions.select_all_regions(
        arr, XC, YC, names=SLICE_REGIONS)
    del arr

for ch in region_arrays:
    x, y, sub = region_arrays[ch]["gulf_stream"]
    print(f"{ch:24s} gulf_stream {sub.shape}  "
          f"min {np.nanmin(sub):.3g}  max {np.nanmax(sub):.3g}")

## Section 5 — Per-field validation

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → components → final; every computed step is shown), rows =
  validation domains.  One shared colour scale per column; land/halo
  NaNs gray; regional boxes on the global row.
- **Figure 2 — PDFs**: same grid.  Probability density; land +
  halo-rim NaNs removed; bins shared per field across domains;
  Eq. Pacific row |lat|>2° filtered for f-normalised fields.
- **Literature comparisons** live in Section 6 at the end of the
  notebook — one subsection PER REFERENCE (a reference may validate
  several fields at once), only where a reference exists.  Images in
  `../literature_figures/`, named
  `{field(s)}_{Citation}_{description}.png`.

In [ ]:
# Section 5 helpers: one call per figure, shared by all fields.
from pathlib import Path

import cartopy.crs as ccrs

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import (
    pipeline_map_grid, mask_wrap_cells, LAND_COLOR,
)
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

# Flat literature directory; files named
# {field}_{Citation}_{description}.png
LIT_DIR = Path("../literature_figures")

# Full dependency chain per field (columns of Figures 1-2), including
# component-level intermediates (gradient / Jacobian components).
CHAINS = {
    "oceTAUX": ["U", "oceTAUX"],
    "oceTAUY": ["V", "oceTAUY"],
    "wind_stress_curl": ["oceTAUX", "oceTAUY", "wind_stress_curl"],
    "ekman_pumping": ["wind_stress_curl", "coriolis_f", "ekman_pumping"],
    "u_ekman": ["oceTAUX", "oceTAUY", "coriolis_f", "u_ekman"],
    "v_ekman": ["oceTAUX", "oceTAUY", "coriolis_f", "v_ekman"],
    "oceQnet": ["oceQnet"],
}

# f-normalised fields: |lat|>2 deg filter on the Eq. Pacific row of
# the PDFs only (maps annotated instead) — plan Clarification 8.
F_NORM = {"ekman_pumping", "u_ekman", "v_ekman"}

# Fields drawn/binned on log scales (∝-squared fields).
LOG_FIELDS = set()

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; shared bins "
            "across domains; log10-x for \u221d-squared fields")


def _pdf_arrays(field):
    """Region arrays for the PDF grid of one field.

    Applies the |lat|>2 deg filter to the Eq. Pacific row for
    f-normalised chain members (filter stated in the figure title).
    Inputs: field (str).  Outputs: dict like region_arrays.
    Generated by LH and Claude
    """
    out = {}
    for f in CHAINS[field]:
        d = dict(region_arrays[f])
        if f in F_NORM:
            x, y, a = d["eq_pacific"]
            d["eq_pacific"] = (x, y,
                               np.where(np.abs(y) > 2.0, a, np.nan))
        out[f] = d
    return out


def figure1_maps(field):
    """Figure 1: dependency-chain map grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | f-normalised: Eq. Pacific extreme near equator "
            "(expected)" if field in F_NORM else "")
    pipeline_map_grid(
        CHAINS[field], region_arrays, CMAP_CFG,
        diverging_cmaps=DIVERGING, log_scale_channels=LOG_FIELDS,
        suptitle=f"Figure 1 \u2014 {field}: pipeline maps{note}",
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: dependency-chain PDF grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | Eq. Pacific: |lat|>2\u00b0 filter (f-normalised)"
            if set(CHAINS[field]) & F_NORM else "")
    pipeline_pdf_grid(
        CHAINS[field], _pdf_arrays(field), CMAP_CFG,
        log10_fields=LOG_FIELDS,
        suptitle=f"Figure 2 \u2014 {field}: {PDF_NOTE}{note}",
    )
    plt.show()


def figure3_literature(field, png_name=None, caption=None,
                       global_view=False):
    """Figure 3: our data vs literature .png for one field.

    Inputs: field (str); png_name (str or None) — file in LIT_DIR;
    caption (str or None) — discussion text; global_view (bool) —
    render OUR panel as a global Robinson map (use when the
    literature figure is a global view) instead of the default
    Gulf Stream regional map.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    region = "global" if global_view else "gulf_stream"

    def _render(ax):
        x, y, arr = region_arrays[field][region]
        if global_view:
            # Same seam/Arctic handling as the Figure 1 global row.
            arr = mask_wrap_cells(x, y, arr)
        ax.set_facecolor(LAND_COLOR)
        im, label = plot_global_field(
            ax, x, y, arr, field, CMAP_CFG,
            log_scale_channels=LOG_FIELDS, diverging_cmaps=DIVERGING,
            transform=ccrs.PlateCarree() if global_view else None,
            add_coastline=global_view,
            coastline_kw={"linewidth": 0.4, "edgecolor": "k"},
        )
        if im is not None:
            plt.colorbar(im, ax=ax, orientation="horizontal",
                         fraction=0.04, pad=0.04, label=label)

    side_by_side(
        _render, LIT_DIR / png_name if png_name else None,
        projection=ccrs.Robinson() if global_view else None,
        caption=caption or ("Discussion: awaiting literature "
                            f"reference for {field}."),
    )
    plt.show()

### 5.1 oceTAUX (eastward wind stress)

Rotated τ east: trades (negative), westerlies (positive).  Companion column U (surface current, native_fields store, same run/date): broadly aligned in wind-driven regimes; WBC jets stand out as NOT wind-aligned.

In [ ]:
figure1_maps("oceTAUX")

In [ ]:
figure2_pdfs("oceTAUX")

### 5.2 oceTAUY (northward wind stress)

Rotated τ north: monsoon/storm-track structure.  Companion column V (surface current): same comparison as U/oceTAUX.

In [ ]:
figure1_maps("oceTAUY")

In [ ]:
figure2_pdfs("oceTAUY")

### 5.3 wind_stress_curl

∇×τ from native gradients of the rotated stress; gyre-scale sign structure.

In [ ]:
figure1_maps("wind_stress_curl")

In [ ]:
figure2_pdfs("wind_stress_curl")

### 5.4 ekman_pumping

w_E = curl(τ)/(ρ₀f); equatorial extremes expected (f→0; PDF row filtered).

In [ ]:
figure1_maps("ekman_pumping")

In [ ]:
figure2_pdfs("ekman_pumping")

### 5.5 u_ekman

Zonal Ekman transport — 90° right of τ in NH, left in SH.

In [ ]:
figure1_maps("u_ekman")

In [ ]:
figure2_pdfs("u_ekman")

### 5.6 v_ekman

Meridional Ekman transport: equatorward under westerlies, poleward under trades.

In [ ]:
figure1_maps("v_ekman")

In [ ]:
figure2_pdfs("v_ekman")

### 5.7 oceQnet (raw, SURF only)

Net surface heat flux (LLC_SURF store), validated as loaded; ocean heat loss over WBCs in November.

In [ ]:
figure1_maps("oceQnet")

In [ ]:
figure2_pdfs("oceQnet")

## Section 6 — Literature comparisons

One subsection per reference (a reference may validate several fields); only fields with published counterparts appear here.

_No literature references supplied yet for this subset — add per-reference subsections here as they become available._

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':24s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    key = hash(sub[finite][::997].tobytes()) if finite.any() else ch
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:24s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

In [ ]:
# Store-vs-live consistency: each final channel (store) must equal
# the function of its live-computed dependencies (Gulf Stream domain;
# land + halo-rim NaNs excluded).  This turns the two-path design
# (finals via RUN->store->LOAD, dependencies via live compute) into
# an explicit pass/fail test of the pipeline plumbing.
from dbof.preprocessing.physical_constants import (
    G, RHO0_REFERENCE, ALPHA, BETA,
)


def _gs(field):
    """Gulf Stream slice of one field.

    Inputs: field (str).  Outputs: 2D np.ndarray.
    Generated by LH and Claude
    """
    return region_arrays[field]["gulf_stream"][2]


CHECKS = {
    "ekman_pumping = curl / (RHO0 * f)":
        (_gs("ekman_pumping"),
         _gs("wind_stress_curl")
         / (RHO0_REFERENCE * _gs("coriolis_f"))),
    "u_ekman = tau_north / (RHO0 * f)":
        (_gs("u_ekman"),
         _gs("oceTAUY") / (RHO0_REFERENCE * _gs("coriolis_f"))),
    "v_ekman = -tau_east / (RHO0 * f)":
        (_gs("v_ekman"),
         -_gs("oceTAUX") / (RHO0_REFERENCE * _gs("coriolis_f"))),
}

REL_TOL = 1e-4       # float32 store vs float64 live recompute
for name, (store_v, recomputed) in CHECKS.items():
    m = np.isfinite(store_v) & np.isfinite(recomputed)
    scale = max(float(np.nanmax(np.abs(recomputed[m]))), 1e-300)
    rel = float(np.nanmax(np.abs(store_v[m] - recomputed[m]))) / scale
    flag = "OK  " if rel < REL_TOL else "FAIL"
    print(f"{flag} {name}  (max rel err {rel:.2e}, n={m.sum()})")
    assert rel < REL_TOL, name
print("\nStore-vs-live consistency: all checks passed.")

**Cross-references** — the DEPTH-pipeline `surface_wind` subset is surface_only and identical in content; it references THIS notebook.  coriolis_f as an output channel is validated in `kinematic.ipynb`.